# Task 2: Sentiment & Thematic Analysis

**Objective**: Quantify review sentiment and identify recurring themes to uncover satisfaction drivers and pain points for each bank (CBE, BOA, Dashen).

## Pipeline Overview
1. Load & preprocess reviews
2. Sentiment analysis (DistilBERT primary + VADER comparison)
3. Aggregate sentiment by bank and star rating
4. TF-IDF keyword extraction per bank
5. Theme assignment (rule-based) + LDA validation
6. Export results CSV

---

### Tool Selection Rationale

| Tool | Type | Why Selected |
|------|------|-------------|
| **DistilBERT** (SST-2) | Transformer | High accuracy on informal review text; handles negation, sarcasm |
| **VADER** | Rule-based | Fast baseline; lexicon-driven compound scores |
| **TextBlob** | Pattern-based | Secondary baseline for polarity comparison |
| **TF-IDF** | Statistical | Surfaces distinctive keywords per bank corpus |
| **LDA** | Topic Model | Unsupervised validation of rule-based themes |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import preprocess_dataframe, normalize_schema
from src.sentiment_analysis import (
    TransformerSentimentAnalyzer,
    VADERSentimentAnalyzer,
    TextBlobSentimentAnalyzer,
    aggregate_sentiment_by_bank,
    aggregate_sentiment_by_rating,
    sentiment_coverage,
)
from src.thematic_analysis import (
    extract_keywords_per_bank,
    assign_themes_to_df,
    summarize_themes_per_bank,
    get_theme_keywords_evidence,
    run_lda_topics,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports ready.')

## 1. Load & Preprocess Reviews

In [ ]:
df = pd.read_csv('../data/raw/reviews.csv')
df = normalize_schema(df)
print(f'Loaded {len(df)} reviews across {df["bank"].nunique()} banks')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# Apply text preprocessing: clean → tokenize → lemmatize
df = preprocess_dataframe(df, text_column='review')
print(f'Preprocessed {len(df)} reviews')
print(f'Sample processed text: {df["processed_content"].iloc[2][:100]}')
df[['review', 'clean_text', 'processed_content']].head(3)

## 2. Sentiment Analysis

### 2.1 DistilBERT Transformer (Primary)
Using `distilbert-base-uncased-finetuned-sst-2-english` for binary sentiment classification.
Reviews with confidence < 0.70 are classified as **neutral**.

In [ ]:
# Initialize and run DistilBERT sentiment analyzer
transformer = TransformerSentimentAnalyzer(neutral_threshold=0.70)
results = transformer.predict_batch(df['review'].tolist(), batch_size=32)

df['sentiment_label'] = [r[0] for r in results]
df['sentiment_score'] = [r[1] for r in results]

# Coverage check
labeled, total, pct = sentiment_coverage(df)
print(f'\nSentiment Coverage: {labeled}/{total} ({pct}%)')
print(f'Label Distribution:\n{df["sentiment_label"].value_counts()}')

### 2.2 VADER Baseline (Comparison)

In [ ]:
vader = VADERSentimentAnalyzer()
vader_results = vader.predict_batch(df['review'].tolist())

df['vader_label'] = [r[0] for r in vader_results]
df['vader_score'] = [r[1] for r in vader_results]

# Agreement between DistilBERT and VADER
agreement = (df['sentiment_label'] == df['vader_label']).mean() * 100
print(f'DistilBERT vs VADER agreement: {agreement:.1f}%')
print(f'\nVADER Label Distribution:\n{df["vader_label"].value_counts()}')

### 2.3 Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DistilBERT distribution
order = ['positive', 'neutral', 'negative']
colors = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
sns.countplot(data=df, x='sentiment_label', order=order, 
              palette=[colors[o] for o in order], ax=axes[0])
axes[0].set_title('DistilBERT Sentiment Distribution', fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

# VADER distribution
sns.countplot(data=df, x='vader_label', order=order,
              palette=[colors[o] for o in order], ax=axes[1])
axes[1].set_title('VADER Sentiment Distribution', fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/sentiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Sentiment Aggregation

### 3.1 By Bank

In [ ]:
bank_agg = aggregate_sentiment_by_bank(df)
bank_agg

In [ ]:
# Stacked bar chart: sentiment distribution per bank
fig, ax = plt.subplots(figsize=(10, 5))
bank_pct = bank_agg[['bank', 'positive_pct', 'neutral_pct', 'negative_pct']].set_index('bank')
bank_pct.plot(kind='bar', stacked=True, color=['#2ecc71', '#95a5a6', '#e74c3c'], ax=ax)
ax.set_title('Sentiment Distribution by Bank (%)', fontweight='bold', fontsize=14)
ax.set_ylabel('Percentage')
ax.set_xlabel('')
ax.legend(['Positive', 'Neutral', 'Negative'], loc='upper right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../data/sentiment_by_bank.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 By Bank × Star Rating

In [ ]:
rating_agg = aggregate_sentiment_by_rating(df)
rating_agg

In [ ]:
# Heatmap: mean sentiment score by bank × star rating
pivot = rating_agg.pivot_table(index='bank', columns='rating', values='mean_score')
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5, ax=ax)
ax.set_title('Mean Sentiment Score by Bank × Star Rating', fontweight='bold', fontsize=14)
ax.set_ylabel('')
ax.set_xlabel('Star Rating')
plt.tight_layout()
plt.savefig('../data/sentiment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Thematic Analysis

### Theme Grouping Logic

Themes are defined as recurring business-relevant categories in user feedback. Our approach:

1. **Keyword Extraction**: TF-IDF (unigrams + bigrams) surfaces the most distinctive terms per bank.
2. **Rule-Based Mapping**: Keywords are matched against a curated dictionary of 5 themes:
   - **Account Access & Login** — login, password, OTP, verification, access
   - **Transaction Performance** — transfer, slow, crash, error, payment
   - **Customer Support** — support, help, complaint, service, agent
   - **UI & App Design** — interface, design, easy, beautiful, smooth
   - **Feature Requests & Bugs** — bug, fix, update, missing, improve
3. **Validation**: LDA topic modeling (unsupervised) cross-checks rule-based assignments.

This deterministic approach ensures interpretability and reproducibility while remaining domain-relevant.

### 4.1 TF-IDF Keyword Extraction

In [ ]:
bank_keywords = extract_keywords_per_bank(df, text_col='processed_content', top_n=30)

for bank, kws in bank_keywords.items():
    print(f'\n--- {bank} Top 15 TF-IDF Keywords ---')
    for term, score in kws[:15]:
        print(f'  {term:25s} {score:.4f}')

### 4.2 Theme Assignment

In [ ]:
df = assign_themes_to_df(df, text_col='processed_content')

print('Theme Distribution (overall):')
print(df['identified_theme'].value_counts())
print(f'\nDistinct themes: {df["identified_theme"].nunique()}')

In [ ]:
# Theme distribution per bank
theme_summary = summarize_themes_per_bank(df)
theme_summary

In [ ]:
# Visualization: Theme distribution per bank
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
banks = df['bank'].unique()

for i, bank in enumerate(sorted(banks)):
    bank_data = theme_summary[theme_summary['bank'] == bank].sort_values('count', ascending=True)
    colors_map = plt.cm.Set2(np.linspace(0, 1, len(bank_data)))
    axes[i].barh(bank_data['identified_theme'], bank_data['count'], color=colors_map)
    axes[i].set_title(f'{bank}', fontweight='bold', fontsize=13)
    axes[i].set_xlabel('Number of Reviews')
    # Add percentage labels
    for j, (_, row) in enumerate(bank_data.iterrows()):
        axes[i].text(row['count'] + 1, j, f"{row['pct']}%", va='center', fontsize=9)

plt.suptitle('Theme Distribution by Bank', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../data/themes_by_bank.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Theme Keyword Evidence

In [ ]:
evidence = get_theme_keywords_evidence(bank_keywords)

for bank, themes in evidence.items():
    print(f'\n=== {bank} ===')
    for theme, kws in themes.items():
        print(f'  {theme}: {", ".join(kws[:6])}')

### 4.4 LDA Topic Modeling (Validation)

In [ ]:
all_texts = df['processed_content'].dropna().tolist()
topics, lda_model, vectorizer = run_lda_topics(all_texts, n_topics=5, top_n_words=10)

print('LDA Topics (unsupervised):')
for i, words in enumerate(topics):
    print(f'  Topic {i+1}: {", ".join(words)}')

## 5. KPI Validation

In [ ]:
print('=' * 50)
print('  KPI VALIDATION REPORT')
print('=' * 50)

# KPI 1: Sentiment scores assigned to 90%+ of reviews
labeled, total, pct = sentiment_coverage(df)
kpi1 = '✓ PASS' if pct >= 90 else '✗ FAIL'
print(f'\n1. Sentiment Coverage: {pct}% ({labeled}/{total}) {kpi1}')

# KPI 2: 3+ distinct themes per bank
print('\n2. Distinct Themes per Bank:')
all_pass = True
for bank in sorted(df['bank'].unique()):
    n_themes = df[df['bank'] == bank]['identified_theme'].nunique()
    status = '✓ PASS' if n_themes >= 3 else '✗ FAIL'
    if n_themes < 3: all_pass = False
    print(f'   {bank}: {n_themes} themes {status}')

# KPI 3: At least 400 reviews with sentiment
kpi3 = '✓ PASS' if labeled >= 400 else '✗ FAIL'
print(f'\n3. Reviews with Sentiment: {labeled} {kpi3}')

# KPI 4: 2+ themes per bank (minimum essential)
print('\n4. Minimum 2 Themes per Bank (Essential):')
for bank in sorted(df['bank'].unique()):
    n_themes = df[df['bank'] == bank]['identified_theme'].nunique()
    status = '✓ PASS' if n_themes >= 2 else '✗ FAIL'
    print(f'   {bank}: {n_themes} themes {status}')

## 6. Save Results

In [ ]:
# Create review_id
df.insert(0, 'review_id', range(1, len(df) + 1))

# Select output columns
output_cols = ['review_id', 'review', 'bank', 'rating', 'date',
               'sentiment_label', 'sentiment_score', 'identified_theme',
               'vader_label', 'vader_score']
output_df = df[[c for c in output_cols if c in df.columns]].copy()
output_df = output_df.rename(columns={'review': 'review_text'})

output_path = '../data/sentiment_themes_results.csv'
output_df.to_csv(output_path, index=False)
print(f'Saved {len(output_df)} rows to {output_path}')
print(f'Columns: {list(output_df.columns)}')
output_df.head()